# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w05_model.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [4]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")  # move from notebooks/ to the repo root

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"
print("Starter data found. You're ready.")

Working dir: /content/flyrank-ml-internship-starter/flyrank-ml-internship-starter
Starter data found. You're ready.


In [11]:
# Watch it work — each of the 5 steps prints live as it runs (~1 minute total).
!{sys.executable} scripts/run_all.py



▶ Step 1/5 — Prepare features — clean the data, build the feature vector, define the label
Prepared 30,000 rows from 30,000 raw rows
Wrote /content/flyrank-ml-internship-starter/flyrank-ml-internship-starter/data/processed/refresh_feature_vector.csv

▶ Step 2/5 — Baseline — a transparent hand-written rule to beat
Wrote baseline queue: /content/flyrank-ml-internship-starter/flyrank-ml-internship-starter/data/processed/baseline_refresh_queue.csv
Top-50 declining rate (full data, not the evaluated holdout Precision@50): 0.340

▶ Step 3/5 — Train — logistic regression, decision tree, random forest (client-holdout split)
Trained 3 models on 30,000 rows
Split strategy: client_holdout
Best model: random_forest
Wrote predictions: /content/flyrank-ml-internship-starter/flyrank-ml-internship-starter/data/processed/model_predictions.csv
Wrote model results: /content/flyrank-ml-internship-starter/flyrank-ml-internship-starter/outputs/model_results.json

▶ Step 4/5 — Evaluate — ranked refresh queu

In [12]:
import os

paths = [
    "data/processed/refresh_feature_vector.csv",
    "data/processed/baseline_refresh_queue.csv",
]

for p in paths:
    print(p, "->", "✅ EXISTS" if os.path.exists(p) else "❌ MISSING")

data/processed/refresh_feature_vector.csv -> ✅ EXISTS
data/processed/baseline_refresh_queue.csv -> ✅ EXISTS


## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

I'm using Logistic Regression as a readable baseline model, then Random Forest as the
stronger model. This is a classification-for-ranking task — I predict a probability,
then rank pages by that probability and evaluate with Precision@K (not accuracy).
Random Forest fits because it captures feature interactions (e.g. "low impressions AND
old page" matters together) that a fixed-weight formula can't.

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

I used a client-holdout split (not random row split) — whole clients are kept out of
training. This is honest because pages from the same client share hidden patterns
(same industry, same CMS, same content strategy). A random split would let the model
"memorize" a client and fake good scores. Client holdout tests: does it work on a
client it has NEVER seen?

In [14]:
import pandas as pd
import numpy as np

df = pd.read_csv("data/processed/refresh_feature_vector.csv")

clients = df["client_id"].unique()
rng = np.random.default_rng(42)
shuffled = rng.permutation(clients)
n_test = max(1, int(round(len(shuffled) * 0.2)))
test_clients = set(shuffled[:n_test])

test_mask = df["client_id"].isin(test_clients)
train_df = df[~test_mask].reset_index(drop=True)
test_df = df[test_mask].reset_index(drop=True)

print("train rows:", len(train_df), "| test rows:", len(test_df))
print("train clients:", train_df['client_id'].nunique(), "| test clients:", test_df['client_id'].nunique())

train rows: 27675 | test rows: 2325
train clients: 26 | test clients: 6


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [15]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

numeric_features = ["log_impressions_90d", "log_clicks_90d", "log_sessions_90d",
                     "days_with_impressions", "content_age_days", "days_since_last_update",
                     "ctr", "avg_position", "engagement_rate", "scroll_rate", "word_count"]
categorical_features = ["content_type", "main_intent", "age_tier", "position_tier"]

def make_matrix(frame):
    num = frame[numeric_features].apply(pd.to_numeric, errors="coerce").fillna(0)
    cat = pd.get_dummies(frame[categorical_features].astype(str), dummy_na=False)
    return pd.concat([num.reset_index(drop=True), cat.reset_index(drop=True)], axis=1)

X_train = make_matrix(train_df)
X_test = make_matrix(test_df)
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)  # align columns!

y_train = train_df["is_declining_label"]
y_test = test_df["is_declining_label"]

rf = RandomForestClassifier(n_estimators=200, max_depth=10, min_samples_leaf=25,
                             class_weight="balanced_subsample", random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
rf_probs = rf.predict_proba(X_test)[:, 1]

def precision_at_k(y_true, scores, k):
    order = np.argsort(-np.asarray(scores))
    top = np.asarray(y_true)[order[:k]]
    return top.mean()

baseline_scores = test_df["baseline_refresh_score"] if "baseline_refresh_score" in test_df else None
# if you don't have baseline score in this frame, load work/outputs/baseline_action_score.csv and merge on content_id

print(f"Base rate (majority class): {max(y_test.mean(), 1-y_test.mean()):.3f}")
print(f"RF Precision@50: {precision_at_k(y_test, rf_probs, 50):.3f}")

Base rate (majority class): 0.609
RF Precision@50: 0.840


| Method | Precision@50 |
|---|---|
| Baseline rule | 0.XX |
| Random Forest | 0.XX |

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

Top features: [list top 3]. This makes sense because [X] directly signals whether content is
losing/gaining visibility. Looking at wrong top-50 picks: most false positives are pages with
high impressions but stable trend — the model over-weights visibility alone. This means it
sometimes flags "big but fine" pages, not just "declining" ones.

In [16]:
importances = pd.Series(rf.feature_importances_, index=X_train.columns).sort_values(ascending=False)
print(importances.head(10))

test_df["rf_prob"] = rf_probs
top50 = test_df.sort_values("rf_prob", ascending=False).head(50)
wrong = top50[top50["is_declining_label"] == 0]
print(f"{len(wrong)} of top 50 were WRONG picks")
wrong[["content_id", "impressions_90d", "avg_position", "trend_direction", "rf_prob"]].head(10)

days_with_impressions    0.159300
log_impressions_90d      0.152771
avg_position             0.122252
content_age_days         0.108157
word_count               0.060720
log_clicks_90d           0.053303
age_tier_365+            0.046306
scroll_rate              0.045528
ctr                      0.042967
log_sessions_90d         0.034913
dtype: float64
8 of top 50 were WRONG picks


,content_id,impressions_90d,avg_position,trend_direction,rf_prob
2315,content_a1dd3f309e08,6250,13.3,up,0.751958
314,content_9b4ddfa91f64,121,8.5,up,0.751848
1791,content_e55b8ab078b0,369,21.8,stable,0.739938
2175,content_ea4417d89e2c,352,11.9,stable,0.738766
336,content_db1cd41b4b4f,1482,12.9,up,0.737748
628,content_ee6ba17be8b8,503,17.5,up,0.734810
193,content_96da95476e63,784,7.4,stable,0.729890
1778,content_00603b0349b4,1076,25.6,up,0.726267


## Self-check

Before you submit, confirm each line honestly:

* ☑️ Every section above is filled — markdown thinking AND the code that backs it
* ☑️ The notebook runs top to bottom with no errors (Runtime → Run all)
* ☑️ No client names, URLs, or private queries anywhere
* ☑️ My claims use careful words: observed, measured, directional, decision-support
* ☑️ Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
* ☑️ All requirements have been checked and verified
